# Fase 1 — Preprocesamiento y Aplanado

Transforma la jerarquía `sessions → levels → rooms` exportada de Firestore
en tres DataFrames planos listos para análisis, y genera las variables derivadas
necesarias para el EDA y el modelo predictivo.

> **Nota sobre el volumen de datos:** Los datos actuales son una muestra inicial
> de desarrollo (11 sesiones). El pipeline está diseñado para escalar automáticamente
> cuando se recojan más sesiones — basta con re-ejecutar `01_extraction.ipynb` y este notebook.

**Entradas:** `data/raw/sessions_raw.json`  
**Salidas:** `data/processed/sessions.parquet`, `levels.parquet`, `rooms.parquet`

## 0. Configuración

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
from preprocessing import run_pipeline, rooms_to_long

RAW_PATH       = Path('..') / 'data' / 'raw'       / 'sessions_raw.json'
PROCESSED_DIR  = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Raw:       {RAW_PATH.resolve()}')
print(f'Processed: {PROCESSED_DIR.resolve()}')

Raw:       D:\DataScience\Proyecto-TFM-HackAndSlash\data\raw\sessions_raw.json
Processed: D:\DataScience\Proyecto-TFM-HackAndSlash\data\processed


## 1. Aplanado de la jerarquía

In [2]:
df_sessions, df_levels, df_rooms = run_pipeline(str(RAW_PATH))

print(f'df_sessions : {df_sessions.shape[0]} filas × {df_sessions.shape[1]} columnas')
print(f'df_levels   : {df_levels.shape[0]} filas × {df_levels.shape[1]} columnas')
print(f'df_rooms    : {df_rooms.shape[0]} filas × {df_rooms.shape[1]} columnas')

df_sessions : 53 filas × 27 columnas
df_levels   : 107 filas × 8 columnas
df_rooms    : 534 filas × 64 columnas


D:\DataScience\Proyecto-TFM-HackAndSlash\notebooks\..\src\preprocessing.py:470: UserWarning: [flag_anomalies] 11 sesiones anómalas detectadas. Revisar columna 'anomaly_reasons'.
  df_sessions = flag_anomalies(df_sessions, df_levels)


In [3]:
# Vista general de sesiones
df_sessions[['sessionId','playerElement','platform','gameVersion',
             'isVictory','totalTimeSecs','totalKills','totalDeaths',
             'levelsCompleted','hasComment']]

,sessionId,playerElement,platform,gameVersion,isVictory,totalTimeSecs,totalKills,totalDeaths,levelsCompleted,hasComment
0,20260512_132734_1952,Fire,Editor,0.1,False,211.10,41,0,2,False
1,20260512_183848_6726,Fire,WebGL,0.1,False,125.00,0,0,0,False
2,20260512_184104_4372,Fire,WebGL,0.1,False,16.00,0,0,0,False
3,20260512_184136_8603,Fire,WebGL,0.1,True,550.00,100,0,4,False
4,20260512_200001_7141,Fire,WebGL,0.1,False,65.40,16,0,1,False
5,20260512_200838_8436,Earth,WebGL,0.1,False,3647.50,41,0,2,False
6,20260512_210947_5724,Fire,WebGL,0.1,False,515.20,71,0,3,False
7,20260512_211828_9481,Wind,WebGL,0.1,False,222.80,16,0,1,False
8,20260513_074033_3474,Fire,WebGL,0.1,False,18.10,0,0,0,True
9,20260513_081206_3246,Water,Editor,0.1,False,60.70,3,0,0,False


## 2. Limpieza — TFM-11

Identificamos y tratamos sesiones problemáticas antes del análisis.

In [4]:
print('=== Valores nulos por columna (sesiones) ===')
nulls = df_sessions.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else 'Sin valores nulos')

print('\n=== Tipos de datos ===')
print(df_sessions.dtypes)

=== Valores nulos por columna (sesiones) ===
Sin valores nulos

=== Tipos de datos ===
sessionId                            object
playerElement                        object
startUnixTime                         int64
endUnixTime                           int64
isVictory                              bool
totalTimeSecs                       float64
totalDeaths                           int64
totalKills                            int64
levelsCompleted                       int64
playerComment                        object
playerCommentContext                 object
platform                             object
gameVersion                          object
eventCount                            int64
startDatetime           datetime64[ns, UTC]
endDatetime             datetime64[ns, UTC]
hasComment                             bool
kd_ratio                            float64
kills_per_min                       float64
time_per_level                      float64
totalLevels                      

In [5]:
# Detectar sesiones sospechosas de ser pruebas o demasiado cortas
MIN_TIME_SECS = 30   # sesiones de menos de 30s probablemente son pruebas
MAX_TIME_SECS = 3600 # sesiones de más de 1 hora son outliers claros

mask_short   = df_sessions['totalTimeSecs'] < MIN_TIME_SECS
mask_long    = df_sessions['totalTimeSecs'] > MAX_TIME_SECS
mask_nokills = (df_sessions['totalKills'] == 0) & (df_sessions['levelsCompleted'] == 0)

print(f'Sesiones muy cortas  (<{MIN_TIME_SECS}s): {mask_short.sum()}')
print(df_sessions[mask_short][['sessionId','totalTimeSecs','totalKills','playerElement']])

print(f'\nSesiones muy largas  (>{MAX_TIME_SECS}s): {mask_long.sum()}')
print(df_sessions[mask_long][['sessionId','totalTimeSecs','totalKills','playerElement']])

print(f'\nSesiones sin kills ni niveles completados: {mask_nokills.sum()}')
print(df_sessions[mask_nokills][['sessionId','totalTimeSecs','totalKills','levelsCompleted']])

Sesiones muy cortas  (<30s): 4
               sessionId  totalTimeSecs  totalKills playerElement
2   20260512_184104_4372          16.00           0          Fire
8   20260513_074033_3474          18.10           0          Fire
26  20260513_181553_8074          11.80           0          Fire
36  20260516_021654_3996           2.80           0          Wind

Sesiones muy largas  (>3600s): 1
              sessionId  totalTimeSecs  totalKills playerElement
5  20260512_200838_8436        3647.50          41         Earth

Sesiones sin kills ni niveles completados: 5
               sessionId  totalTimeSecs  totalKills  levelsCompleted
1   20260512_183848_6726         125.00           0                0
2   20260512_184104_4372          16.00           0                0
8   20260513_074033_3474          18.10           0                0
26  20260513_181553_8074          11.80           0                0
36  20260516_021654_3996           2.80           0                0


In [6]:
# Detectar niveles incompletos
niveles_incompletos = df_levels[df_levels['incomplete'] == True]
print(f'Niveles incompletos: {len(niveles_incompletos)}')
print(niveles_incompletos[['sessionId','levelId','kills','deaths','timeSecs']])

Niveles incompletos: 27
                sessionId levelId  kills  deaths  timeSecs
2    20260512_183848_6726  Level1      0       0      0.00
3    20260512_184104_4372  Level1      0       0      0.00
15   20260513_074033_3474  Level1      0       0      0.00
16   20260513_081206_3246  Level1      3       0      0.00
17   20260513_082947_4162  Level1      5       0      0.00
27   20260513_112014_1682  Level2     11       0      0.00
51   20260513_170401_4209  Level1      4       0      0.00
52   20260513_171151_4601  Level1      9       0      0.00
54   20260513_173548_1135  Level2      7       0      0.00
58   20260513_180319_6726  Level4      0       0      0.00
59   20260513_181907_4441  Level1      2       0      0.00
64   20260514_095957_5124  Level1      2       0      0.00
65   20260514_141421_7239  Level1     14       0      0.00
70   20260516_014525_8622  Level2      0       0      0.00
71   20260516_020407_1739  Level1      9       1      0.00
72   20260516_020645_6058  Level

In [7]:
from cleaning import flag_suspicious_sessions

# Delegar toda la lógica de marcado a cleaning.py (incluye Editor + 120s + no_activity + >3600s)
df_sessions = flag_suspicious_sessions(df_sessions)

print(f'Sesiones totales    : {len(df_sessions)}')
print(f'Sesiones sospechosas: {df_sessions["is_suspicious"].sum()}')
print(f'Sesiones limpias    : {(~df_sessions["is_suspicious"]).sum()}')
print(f'Plataformas en sospechosas: {df_sessions[df_sessions["is_suspicious"]]["platform"].value_counts().to_dict()}')
print(f'Plataformas en limpias    : {df_sessions[~df_sessions["is_suspicious"]]["platform"].value_counts().to_dict()}')

Sesiones totales    : 53
Sesiones sospechosas: 18
Sesiones limpias    : 35
Plataformas en sospechosas: {'WebGL': 13, 'Editor': 5}
Plataformas en limpias    : {'WebGL': 35}


> **Criterio:** Con el volumen actual (11 sesiones) no descartamos ninguna.
> El flag `is_suspicious` permite filtrar en análisis que requieran datos de calidad.

## 3. Variables calculadas — TFM-12

In [8]:
# Variables derivadas de sesión
cols_features = ['sessionId','playerElement','isVictory',
                 'kd_ratio','kills_per_min','time_per_level',
                 'completion_rate','totalRooms','totalCast']

print('=== Features de sesión ===')
print(df_sessions[cols_features].to_string())

=== Features de sesión ===
               sessionId playerElement  isVictory  kd_ratio  kills_per_min  time_per_level  completion_rate  totalRooms  totalCast
0   20260512_132734_1952          Fire      False     41.00          11.65          105.55             1.00          11        101
1   20260512_183848_6726          Fire      False      0.00           0.00          125.00             0.00           1          3
2   20260512_184104_4372          Fire      False      0.00           0.00           16.00             0.00           1          1
3   20260512_184136_8603          Fire       True    100.00          10.91          137.50             1.00          24        263
4   20260512_200001_7141          Fire      False     16.00          14.68           65.40             1.00           5         27
5   20260512_200838_8436         Earth      False     41.00           0.67         1823.75             1.00          11        237
6   20260512_210947_5724          Fire      False     71

In [9]:
# Variables derivadas de sala
cols_room_features = ['sessionId','levelId','roomId',
                      'total_kills_room','total_cast_room','total_miss_room',
                      'cast_accuracy','kills_per_sec','deaths','damageTaken']

print('=== Features de sala (primeras 10) ===')
print(df_rooms[cols_room_features].head(10).to_string())

=== Features de sala (primeras 10) ===
              sessionId levelId     roomId  total_kills_room  total_cast_room  total_miss_room  cast_accuracy  kills_per_sec  deaths  damageTaken
0  20260512_132734_1952  Level1    Room_01              2.00             5.00             0.00           1.00           0.24       0         0.00
1  20260512_132734_1952  Level1    Room_02              3.00             9.00             0.00           1.00           0.25       0        20.00
2  20260512_132734_1952  Level1    Room_03              4.00             7.00             0.00           1.00           0.28       0        48.00
3  20260512_132734_1952  Level1    Room_04              5.00             6.00             0.00           1.00           0.49       0       118.00
4  20260512_132734_1952  Level1  Room_Boss              2.00             5.00             1.00           0.80           0.19       0       126.00
5  20260512_132734_1952  Level2    Room_01              3.00            13.00        

In [10]:
# Inventario de hechizos y enemigos detectados
spell_types  = sorted({c.split('_',1)[1] for c in df_rooms.columns if c.startswith('cast_')})
enemy_types  = sorted({c.split('_',1)[1] for c in df_rooms.columns if c.startswith('kills_')})
status_types = sorted({c.split('_',1)[1] for c in df_rooms.columns if c.startswith('status_')})

print(f'Hechizos detectados ({len(spell_types)}): {spell_types}')
print(f'Enemigos detectados ({len(enemy_types)}): {enemy_types}')
print(f'Efectos de estado  ({len(status_types)}): {status_types}')

Hechizos detectados (7): ['AOE', 'Aura', 'Beam', 'Blast', 'Projectile', 'Shield', 'accuracy']
Enemigos detectados (14): ['Barbarian1HBoss', 'Barbarian1Hand', 'Barbarian2HBoss', 'Barbarian2Hand', 'Knight1H', 'Knight2H', 'KnightBossBlack', 'KnightBossGold', 'RangerBow', 'RangerBowBoss', 'RangerCrossbow', 'Rogue', 'Rogue_Hooded', 'per_sec']
Efectos de estado  (4): ['Burn', 'Knockback', 'Slow', 'Stun']


In [11]:
# Formato long para hechizos (útil para comparar uso entre tipos)
df_cast_long = rooms_to_long(df_rooms, 'cast')
print(f'Lanzamientos por hechizo (formato long): {len(df_cast_long)} filas')
print(df_cast_long.groupby('cast_type')['value'].sum().sort_values(ascending=False))

Lanzamientos por hechizo (formato long): 1967 filas
cast_type
Projectile   6273.00
Blast        1816.00
accuracy      457.21
Beam          431.00
AOE           213.00
Shield         13.00
Aura            3.00
Name: value, dtype: float64


In [12]:
# Formato long para kills por enemigo
df_kills_long = rooms_to_long(df_rooms, 'kills')
print(f'Kills por enemigo (formato long): {len(df_kills_long)} filas')
print(df_kills_long.groupby('kills_type')['value'].sum().sort_values(ascending=False))

Kills por enemigo (formato long): 1787 filas
kills_type
Rogue             411.00
Knight2H          387.00
RangerCrossbow    235.00
RangerBow         234.00
Knight1H          190.00
Barbarian1Hand    175.00
Barbarian2Hand    167.00
per_sec            89.28
Barbarian2HBoss    54.00
KnightBossGold     33.00
Barbarian1HBoss    30.00
KnightBossBlack    30.00
Rogue_Hooded       12.00
RangerBowBoss      10.00
Name: value, dtype: float64


## 4. Estadísticas descriptivas rápidas

In [13]:
print('=== Sesiones por elemento ===')
print(df_sessions.groupby('playerElement').agg(
    sesiones=('sessionId','count'),
    victorias=('isVictory','sum'),
    kills_media=('totalKills','mean'),
    tiempo_medio=('totalTimeSecs','mean')
).round(1))

=== Sesiones por elemento ===
               sesiones  victorias  kills_media  tiempo_medio
playerElement                                                
Earth                 9          3        47.60        958.70
Fire                 21          4        34.40        303.90
Water                13          2        34.00        710.30
Wind                 10          1        41.70        654.20


In [14]:
print('=== Dificultad por nivel (deaths + damageTaken medios) ===')
print(df_levels.groupby('levelId').agg(
    sesiones=('sessionId','count'),
    muertes_media=('deaths','mean'),
    daño_media=('damageTaken','mean'),
    tiempo_medio=('timeSecs','mean'),
    intentos_medio=('attempt','mean')
).round(1).sort_values('levelId'))

=== Dificultad por nivel (deaths + damageTaken medios) ===
         sesiones  muertes_media  daño_media  tiempo_medio  intentos_medio
levelId                                                                   
Level1         52           0.20      217.10        113.20            2.00
Level2         27           0.10      190.60        218.90            1.50
Level3         17           0.20      168.50        230.40            1.90
Level4         11           0.10      193.50        223.40            1.70


## 5. Guardar en data/processed/

In [15]:
# Guardar como Parquet (más eficiente que CSV para datos con muchas columnas)
df_sessions.to_parquet(PROCESSED_DIR / 'sessions.parquet', index=False)
df_levels.to_parquet(  PROCESSED_DIR / 'levels.parquet',   index=False)
df_rooms.to_parquet(   PROCESSED_DIR / 'rooms.parquet',    index=False)

# También CSV para inspección manual
df_sessions.to_csv(PROCESSED_DIR / 'sessions.csv', index=False)
df_levels.to_csv(  PROCESSED_DIR / 'levels.csv',   index=False)
df_rooms.to_csv(   PROCESSED_DIR / 'rooms.csv',    index=False)

print('✓ Archivos guardados en data/processed/')
for f in sorted((PROCESSED_DIR).iterdir()):
    print(f'  {f.name}: {f.stat().st_size/1024:.1f} KB')

✓ Archivos guardados en data/processed/


  levels.csv: 5.5 KB
  levels.parquet: 6.9 KB
  rooms.csv: 163.6 KB
  rooms.parquet: 60.9 KB
  sessions.csv: 12.0 KB
  sessions.parquet: 21.8 KB


## 6. Conclusiones

- **Datos actuales:** 11 sesiones, 18 niveles, 82 salas — muestra de desarrollo, insuficiente para modelos estadísticos robustos. Se actualizará cuando haya más testers.
- **Limpieza:** No se descarta ninguna sesión. Flag `is_suspicious` para las 4 sesiones muy cortas o outliers de tiempo.
- **Columnas derivadas creadas:** `kd_ratio`, `kills_per_min`, `time_per_level`, `completion_rate`, `totalRooms`, `totalCast`, `cast_accuracy`, `kills_per_sec`.
- **Hechizos:** Projectile y Blast son los más usados; AOE y Beam aparecen puntualmente.
- **Enemigos:** Variedad de 15+ tipos; los boss tienen kills bajas (esperado).

**Siguiente paso:** `03_eda.ipynb` — visualizaciones exploratorias.